In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "torchaudio", "soundfile", "librosa", "resampy",
                "einops", "tqdm", "matplotlib", "pandas"], check=True)


In [ ]:
import os, random, json
from collections import defaultdict
import torch
import torchaudio
import soundfile as sf
import numpy as np
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

SR = 8000
SEGMENT_S = 4.0
SEGMENT_LEN = int(SR * SEGMENT_S)
SPEAKER_COUNTS = [2, 3, 4, 5]
N_TRAIN_PER_COUNT = 1500
N_TEST_PER_COUNT = 200
TEST_SPEAKER_FRACTION = 0.2

DATA_DIR = "./data"
LIBRISPEECH_ROOT = os.path.join(DATA_DIR, "librispeech")
MIX_ROOT = os.path.join(DATA_DIR, "mixtures")
METADATA_PATH = os.path.join(DATA_DIR, "metadata.json")

os.makedirs(LIBRISPEECH_ROOT, exist_ok=True)
os.makedirs(MIX_ROOT, exist_ok=True)

corpus = torchaudio.datasets.LIBRISPEECH(LIBRISPEECH_ROOT, url="dev-clean", download=True)

speaker_to_waveforms = defaultdict(list)
for idx in tqdm(range(len(corpus)), desc="loading corpus"):
    waveform, sample_rate, _, speaker_id, _, _ = corpus[idx]
    waveform = torchaudio.functional.resample(waveform, sample_rate, SR)
    waveform = waveform.mean(dim=0)
    if waveform.shape[0] >= SEGMENT_LEN:
        speaker_to_waveforms[speaker_id].append(waveform.numpy().astype(np.float32))

speaker_ids = sorted(speaker_to_waveforms.keys())
random.shuffle(speaker_ids)
n_test_speakers = max(SPEAKER_COUNTS[-1], int(len(speaker_ids) * TEST_SPEAKER_FRACTION))
test_speakers = speaker_ids[:n_test_speakers]
train_speakers = speaker_ids[n_test_speakers:]

def random_crop(wav):
    if wav.shape[0] == SEGMENT_LEN:
        return wav
    start = random.randint(0, wav.shape[0] - SEGMENT_LEN)
    return wav[start:start + SEGMENT_LEN]

def make_mixture(speaker_pool, n_spk):
    chosen_speakers = random.sample(speaker_pool, n_spk)
    raw_sources = [random_crop(random.choice(speaker_to_waveforms[sid])) for sid in chosen_speakers]
    gains = [random.uniform(0.7, 1.3) for _ in raw_sources]
    scaled_sources = [g * s for g, s in zip(gains, raw_sources)]
    mixture = np.sum(scaled_sources, axis=0)
    peak = np.max(np.abs(mixture))
    norm = 0.9 / peak if peak > 1e-8 else 1.0
    mixture = mixture * norm
    scaled_sources = [s * norm for s in scaled_sources]
    return mixture.astype(np.float32), [s.astype(np.float32) for s in scaled_sources]

metadata = []
for split, speaker_pool, n_per_count in [
    ("train", train_speakers, N_TRAIN_PER_COUNT),
    ("test", test_speakers, N_TEST_PER_COUNT),
]:
    for n_spk in SPEAKER_COUNTS:
        out_dir = os.path.join(MIX_ROOT, f"{n_spk}spk", split)
        os.makedirs(out_dir, exist_ok=True)
        for i in tqdm(range(n_per_count), desc=f"{split} {n_spk}spk"):
            mixture, sources = make_mixture(speaker_pool, n_spk)
            mix_path = os.path.join(out_dir, f"mix_{i:05d}.wav")
            sf.write(mix_path, mixture, SR)
            source_paths = []
            for si, s in enumerate(sources):
                s_path = os.path.join(out_dir, f"mix_{i:05d}_s{si}.wav")
                sf.write(s_path, s, SR)
                source_paths.append(s_path)
            metadata.append({
                "mixture": mix_path,
                "sources": source_paths,
                "n_speakers": n_spk,
            })

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f)

print(len(metadata), "mixtures generated")


In [ ]:
import json
with open("./data/metadata.json") as f:
    metadata = json.load(f)
print(len(metadata), "examples loaded")


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, random, itertools, math, os
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
class MixtureDataset(Dataset):
    def __init__(self, metadata, split, n_speakers, sr=8000, segment_s=4.0):
        self.items = [m for m in metadata if m["n_speakers"] == n_speakers and f"/{split}/" in m["mixture"]]
        self.sr = sr
        self.segment_len = int(segment_s * sr)

    def __len__(self):
        return len(self.items)

    def _load(self, path):
        wav, _ = sf.read(path)
        return wav.astype(np.float32)

    def __getitem__(self, idx):
        item = self.items[idx]
        mix = self._load(item["mixture"])
        sources = [self._load(p) for p in item["sources"]]

        L = len(mix)
        if L > self.segment_len:
            start = random.randint(0, L - self.segment_len)
            mix = mix[start:start + self.segment_len]
            sources = [s[start:start + self.segment_len] for s in sources]
        else:
            pad = self.segment_len - L
            mix = np.pad(mix, (0, pad))
            sources = [np.pad(s, (0, pad)) for s in sources]

        return torch.from_numpy(mix), torch.stack([torch.from_numpy(s) for s in sources])

_ds = MixtureDataset(metadata, "train", 2)
print(f"2-speaker train examples: {len(_ds)}")
_mix, _src = _ds[0]
print("mix shape:", _mix.shape, "sources shape:", _src.shape)


In [ ]:
class BandSpecialist(nn.Module):
    def __init__(self, band_width, hidden=128):
        super().__init__()
        self.in_proj = nn.Linear(band_width, hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True, bidirectional=True)
        self.out_proj = nn.Linear(hidden * 2, hidden)

    def forward(self, x):
        h = self.in_proj(x)
        h, _ = self.gru(h)
        h = self.out_proj(h)
        return h

class CrossBandFusion(nn.Module):
    def __init__(self, hidden, n_bands):
        super().__init__()
        self.summary_proj = nn.Linear(hidden, hidden)
        self.fusion_gru = nn.GRU(hidden * n_bands, hidden, batch_first=True, bidirectional=True)
        self.film_scale = nn.ModuleList([nn.Linear(hidden * 2, hidden) for _ in range(n_bands)])
        self.film_shift = nn.ModuleList([nn.Linear(hidden * 2, hidden) for _ in range(n_bands)])

    def forward(self, band_feats):
        summaries = [self.summary_proj(f) for f in band_feats]
        concat = torch.cat(summaries, dim=-1)
        fused, _ = self.fusion_gru(concat)

        conditioned = []
        for i, f in enumerate(band_feats):
            scale = torch.sigmoid(self.film_scale[i](fused))
            shift = self.film_shift[i](fused)
            conditioned.append(f * scale + shift)
        return conditioned

class BandRefineStep(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.gru = nn.GRU(hidden, hidden, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(hidden * 2, hidden)
        self.norm = nn.LayerNorm(hidden)

    def forward(self, x):
        residual = x
        h, _ = self.gru(x)
        h = self.proj(h)
        return self.norm(residual + h)

class IterationBlock(nn.Module):
    def __init__(self, hidden, n_bands):
        super().__init__()
        self.fusion = CrossBandFusion(hidden, n_bands)
        self.refiners = nn.ModuleList([BandRefineStep(hidden) for _ in range(n_bands)])

    def forward(self, band_feats):
        band_feats = self.fusion(band_feats)
        band_feats = [refine(f) for refine, f in zip(self.refiners, band_feats)]
        return band_feats

class BandMaskHead(nn.Module):
    def __init__(self, hidden, band_width, n_speakers):
        super().__init__()
        self.proj = nn.Linear(hidden, band_width * n_speakers)
        self.n_speakers = n_speakers
        self.band_width = band_width

    def forward(self, x):
        B, T, _ = x.shape
        m = self.proj(x)
        m = m.view(B, T, self.n_speakers, self.band_width)
        m = m.permute(0, 2, 1, 3)
        return torch.sigmoid(m)

class FrequencyBandEnsemble(nn.Module):
    def __init__(self, n_speakers, n_fft=256, hop=64, hidden=128, band_edges=None, n_repeats=3):
        super().__init__()
        self.n_speakers = n_speakers
        self.n_fft = n_fft
        self.hop = hop
        self.register_buffer("window", torch.hann_window(n_fft))

        n_freq = n_fft // 2 + 1
        if band_edges is None:
            third = n_freq // 3
            band_edges = [(0, third), (third, 2 * third), (2 * third, n_freq)]
        self.band_edges = band_edges

        self.specialists = nn.ModuleList([BandSpecialist(hi - lo, hidden) for (lo, hi) in band_edges])
        self.iteration_blocks = nn.ModuleList([IterationBlock(hidden, len(band_edges)) for _ in range(n_repeats)])
        self.mask_heads = nn.ModuleList([BandMaskHead(hidden, hi - lo, n_speakers) for (lo, hi) in band_edges])

    def forward(self, mixture):
        length = mixture.shape[-1]
        spec = torch.stft(
            mixture, n_fft=self.n_fft, hop_length=self.hop,
            win_length=self.n_fft, window=self.window, return_complex=True
        )
        mag = spec.abs().transpose(1, 2)

        band_inputs = [mag[:, :, lo:hi] for (lo, hi) in self.band_edges]
        band_feats = [net(x) for net, x in zip(self.specialists, band_inputs)]

        for block in self.iteration_blocks:
            band_feats = block(band_feats)

        band_masks = [head(f) for head, f in zip(self.mask_heads, band_feats)]

        full_mask = torch.cat(band_masks, dim=-1)
        full_mask = full_mask.permute(0, 1, 3, 2)

        spec_expanded = spec.unsqueeze(1)
        masked_spec = full_mask * spec_expanded

        B, C, Fbins, T = masked_spec.shape
        masked_flat = masked_spec.reshape(B * C, Fbins, T)
        wav = torch.istft(
            masked_flat, n_fft=self.n_fft, hop_length=self.hop,
            win_length=self.n_fft, window=self.window, length=length
        )
        wav = wav.reshape(B, C, length)
        return wav

print("Model classes defined (v2: iterative refinement).")


In [ ]:
def si_sdr_safe(estimate, target, eps=1e-8, min_target_energy=1e-4, soft_bound=100.0):
    target_energy = torch.sum(target ** 2, dim=-1)
    valid = (target_energy > min_target_energy).float()

    target_c = target - target.mean(dim=-1, keepdim=True)
    estimate_c = estimate - estimate.mean(dim=-1, keepdim=True)
    s_target = (torch.sum(estimate_c * target_c, dim=-1, keepdim=True) * target_c) / \
               (torch.sum(target_c ** 2, dim=-1, keepdim=True) + eps)
    e_noise = estimate_c - s_target
    ratio = (torch.sum(s_target ** 2, dim=-1) + eps) / (torch.sum(e_noise ** 2, dim=-1) + eps)
    score = 10 * torch.log10(ratio + eps)
    score = soft_bound * torch.tanh(score / soft_bound)
    return score, valid

def pit_si_sdr_loss(estimates, targets):
    B, C, T = estimates.shape
    perms = list(itertools.permutations(range(C)))
    best = None
    for perm in perms:
        scores, valids = [], []
        for c in range(C):
            s, v = si_sdr_safe(estimates[:, c], targets[:, perm[c]])
            scores.append(s)
            valids.append(v)
        scores = torch.stack(scores, dim=1)
        valids = torch.stack(valids, dim=1)
        denom = valids.sum(dim=1).clamp(min=1.0)
        mean_score = (scores * valids).sum(dim=1) / denom
        best = mean_score if best is None else torch.maximum(best, mean_score)
    return -best.mean()

print("Loss functions defined.")


In [ ]:
model_smoke = FrequencyBandEnsemble(n_speakers=2).to(device)
dummy_mix = torch.randn(2, 32000, device=device)
dummy_src = torch.randn(2, 2, 32000, device=device)

out = model_smoke(dummy_mix)
print("Output shape:", out.shape, "(expected (2, 2, 32000))")
assert out.shape == dummy_src.shape

loss = pit_si_sdr_loss(out, dummy_src)
print("Smoke test loss (random init vs random target, expect a large but FINITE number):", loss.item())
assert torch.isfinite(loss), "Loss is not finite -- stop and debug before real training!"

loss.backward()
grad_norms = [p.grad.norm().item() for p in model_smoke.parameters() if p.grad is not None]
n_zero_grad = sum(1 for g in grad_norms if g == 0.0)
print(f"Mean grad norm: {sum(grad_norms) / len(grad_norms):.4f}  |  params with exactly-zero grad: {n_zero_grad}/{len(grad_norms)}")
assert n_zero_grad == 0, "Some parameters have exactly zero gradient -- likely a dead unit or an over-aggressive clamp somewhere. Stop and debug."
print("Smoke test passed: finite loss, every parameter received a nonzero gradient.")

del model_smoke, dummy_mix, dummy_src, out, loss


In [ ]:
import math

SPEAKER_COUNTS = [2, 3, 4, 5]
EPOCHS = 40
BATCH_SIZE = 8
LR_MAX = 2e-4
WARMUP_STEPS = 300
LOG_EVERY = 50
CKPT_DIR = "./checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def make_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def train_one_model(n_spk):
    ckpt_path = f"{CKPT_DIR}/band_ensemble_{n_spk}spk.pt"
    model = FrequencyBandEnsemble(n_speakers=n_spk).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX)

    train_ds = MixtureDataset(metadata, "train", n_spk)
    if len(train_ds) == 0:
        print(f"[{n_spk}spk] no training data found, skipping.")
        return model
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    total_steps = EPOCHS * len(train_loader)
    scheduler = make_scheduler(optimizer, total_steps, WARMUP_STEPS)

    start_epoch, global_step = 0, 0
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"]
        global_step = ckpt["global_step"]
        print(f"[{n_spk}spk] resumed from epoch {start_epoch}, global_step {global_step}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        running, n_batches = 0.0, 0
        for step, (mix, sources) in enumerate(train_loader):
            mix, sources = mix.to(device), sources.to(device)
            estimates = model(mix)
            loss = pit_si_sdr_loss(estimates, sources)

            if not torch.isfinite(loss):
                print(f"[{n_spk}spk] epoch {epoch} step {step}: non-finite loss, skipping this batch")
                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
            global_step += 1

            running += loss.item()
            n_batches += 1
            if step % LOG_EVERY == 0:
                current_lr = optimizer.param_groups[0]["lr"]
                print(f"  [{n_spk}spk] epoch {epoch} step {step} loss {loss.item():.3f} (lr={current_lr:.2e})")

        avg_loss = running / max(1, n_batches)
        print(f"[{n_spk}spk] epoch {epoch} avg loss {avg_loss:.3f}")

        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch + 1,
            "global_step": global_step,
        }, ckpt_path)

    return model

models = {}
for n_spk in SPEAKER_COUNTS:
    print(f"=== Training {n_spk}-speaker model ===")
    models[n_spk] = train_one_model(n_spk)

print("All stages complete.")


In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn = nn.Linear(hidden, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x).squeeze(-1), dim=-1)
        return torch.sum(x * weights.unsqueeze(-1), dim=1)

class SpeakerCountClassifier(nn.Module):
    def __init__(self, n_fft=256, hop=64, hidden=192, count_options=(2, 3, 4, 5)):
        super().__init__()
        self.n_fft = n_fft
        self.hop = hop
        self.count_options = count_options
        self.register_buffer("window", torch.hann_window(n_fft))
        n_freq = n_fft // 2 + 1
        self.in_proj = nn.Linear(n_freq, hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True, bidirectional=True)
        self.pool_proj = nn.Linear(hidden * 2, hidden)
        self.pool = AttentionPool(hidden)
        self.head = nn.Linear(hidden, len(count_options))

    def forward(self, mixture):
        spec = torch.stft(
            mixture, n_fft=self.n_fft, hop_length=self.hop,
            win_length=self.n_fft, window=self.window, return_complex=True
        )
        mag = spec.abs().transpose(1, 2)
        mag = torch.log1p(mag)
        h = self.in_proj(mag)
        h, _ = self.gru(h)
        h = self.pool_proj(h)
        pooled = self.pool(h)
        return self.head(pooled)

    @torch.no_grad()
    def predict_count(self, mixture):
        logits = self.forward(mixture)
        idx = logits.argmax(dim=-1)
        options = torch.tensor(self.count_options, device=mixture.device)
        return options[idx]

class SpeakerCountDataset(Dataset):
    def __init__(self, metadata, split, sr=8000, segment_s=4.0, count_options=(2, 3, 4, 5)):
        self.items = [m for m in metadata if f"/{split}/" in m["mixture"] and m["n_speakers"] in count_options]
        self.sr = sr
        self.segment_len = int(segment_s * sr)
        self.count_options = count_options

    def __len__(self):
        return len(self.items)

    def _load(self, path):
        wav, _ = sf.read(path)
        return wav.astype(np.float32)

    def __getitem__(self, idx):
        item = self.items[idx]
        mix = self._load(item["mixture"])
        L = len(mix)
        if L > self.segment_len:
            start = random.randint(0, L - self.segment_len)
            mix = mix[start:start + self.segment_len]
        else:
            mix = np.pad(mix, (0, self.segment_len - L))
        label = self.count_options.index(item["n_speakers"])
        return torch.from_numpy(mix), label

print("Speaker-count classifier classes defined (log-compressed input, attention pooling).")


In [ ]:
COUNT_CLF_EPOCHS = 60
COUNT_CLF_BATCH_SIZE = 16
COUNT_CLF_LR = 1e-3
COUNT_CLF_WARMUP = 200
COUNT_CLF_CKPT = f"{CKPT_DIR}/speaker_count_classifier.pt"

def train_count_classifier():
    model = SpeakerCountClassifier().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=COUNT_CLF_LR)

    train_ds = SpeakerCountDataset(metadata, "train")
    train_loader = DataLoader(train_ds, batch_size=COUNT_CLF_BATCH_SIZE, shuffle=True, num_workers=2)
    total_steps = COUNT_CLF_EPOCHS * len(train_loader)
    scheduler = make_scheduler(optimizer, total_steps, COUNT_CLF_WARMUP)

    start_epoch = 0
    if os.path.exists(COUNT_CLF_CKPT):
        ckpt = torch.load(COUNT_CLF_CKPT, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"]
        print(f"[count-clf] resumed from epoch {start_epoch}")

    for epoch in range(start_epoch, COUNT_CLF_EPOCHS):
        model.train()
        running, correct, total = 0.0, 0, 0
        for mix, label in train_loader:
            mix, label = mix.to(device), label.to(device)
            logits = model(mix)
            loss = F.cross_entropy(logits, label)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()

            running += loss.item()
            correct += (logits.argmax(-1) == label).sum().item()
            total += label.size(0)

        print(f"[count-clf] epoch {epoch} loss {running / len(train_loader):.3f} train_acc {correct / total:.3f}")
        torch.save({
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "epoch": epoch + 1,
        }, COUNT_CLF_CKPT)

    return model

count_model = train_count_classifier()


In [ ]:
@torch.no_grad()
def evaluate_count_classifier(count_model, metadata, split="test", batch_size=16):
    ds = SpeakerCountDataset(metadata, split)
    loader = DataLoader(ds, batch_size=batch_size)
    count_model.eval()
    correct_by_true = {c: 0 for c in count_model.count_options}
    total_by_true = {c: 0 for c in count_model.count_options}
    for mix, label in loader:
        mix, label = mix.to(device), label.to(device)
        pred_idx = count_model(mix).argmax(-1)
        for true_idx, p_idx in zip(label.tolist(), pred_idx.tolist()):
            true_count = count_model.count_options[true_idx]
            total_by_true[true_count] += 1
            if true_idx == p_idx:
                correct_by_true[true_count] += 1
    print("=== Speaker-count classifier accuracy (test split) ===")
    for c in count_model.count_options:
        if total_by_true[c] > 0:
            acc = correct_by_true[c] / total_by_true[c]
            print(f"true count {c}: {acc:.1%} correctly identified ({correct_by_true[c]}/{total_by_true[c]})")

evaluate_count_classifier(count_model, metadata)


In [ ]:
@torch.no_grad()
def separate_blind(mixture, count_model, sep_models):
    predicted_counts = count_model.predict_count(mixture)
    outputs = []
    for i in range(mixture.shape[0]):
        n_pred = int(predicted_counts[i].item())
        model = sep_models[n_pred]
        single_mix = mixture[i:i+1]
        est = model(single_mix)
        outputs.append((n_pred, est[0]))
    return outputs

print("Blind pipeline function defined.")


In [ ]:
@torch.no_grad()
def evaluate_blind(count_model, sep_models, metadata, split="test", mismatch_penalty_db=0.0):
    results = {c: {"conditional": [], "blind": []} for c in SPEAKER_COUNTS}
    count_model.eval()

    for true_count in SPEAKER_COUNTS:
        ds = MixtureDataset(metadata, split, true_count)
        if len(ds) == 0:
            continue
        loader = DataLoader(ds, batch_size=4)
        for mix, sources in loader:
            mix, sources = mix.to(device), sources.to(device)
            predicted_counts = count_model.predict_count(mix)

            for i in range(mix.shape[0]):
                n_pred = int(predicted_counts[i].item())
                if n_pred == true_count:
                    est = sep_models[n_pred](mix[i:i+1])[0]
                    C = true_count
                    perms = list(itertools.permutations(range(C)))
                    best = None
                    for perm in perms:
                        s_list, v_list = [], []
                        for c in range(C):
                            s, v = si_sdr_safe(est[c:c+1], sources[i, perm[c]:perm[c]+1])
                            s_list.append(s); v_list.append(v)
                        s_stack = torch.cat(s_list)
                        v_stack = torch.cat(v_list)
                        denom = v_stack.sum().clamp(min=1.0)
                        mean_score = (s_stack * v_stack).sum() / denom
                        best = mean_score if best is None else torch.maximum(best, mean_score)
                    score = best.item()
                    results[true_count]["conditional"].append(score)
                    results[true_count]["blind"].append(score)
                else:
                    results[true_count]["blind"].append(mismatch_penalty_db)

    print("=== Blind evaluation results (test split) ===")
    print(f"{'speakers':>10} | {'conditional SI-SDR':>20} | {'blind SI-SDR':>14} | {'count acc':>10}")
    for c in SPEAKER_COUNTS:
        cond = results[c]["conditional"]
        blind = results[c]["blind"]
        if len(blind) == 0:
            continue
        cond_str = f"{sum(cond) / len(cond):>17.2f} dB" if cond else f"{'N/A (0 correct)':>20}"
        blind_avg = sum(blind) / len(blind)
        acc = len(cond) / len(blind)
        print(f"{c:>10} | {cond_str} | {blind_avg:>11.2f} dB | {acc:>9.1%}")

    return results

blind_results = evaluate_blind(count_model, models, metadata)


In [ ]:
@torch.no_grad()
def evaluate_oracle(model, n_spk, split="test", batch_size=4):
    ds = MixtureDataset(metadata, split, n_spk)
    if len(ds) == 0:
        return None
    loader = DataLoader(ds, batch_size=batch_size)
    model.eval()
    scores = []
    for mix, sources in loader:
        mix, sources = mix.to(device), sources.to(device)
        estimates = model(mix)
        C = n_spk
        perms = list(itertools.permutations(range(C)))
        best = None
        for perm in perms:
            s_list, v_list = [], []
            for c in range(C):
                s, v = si_sdr_safe(estimates[:, c], sources[:, perm[c]])
                s_list.append(s); v_list.append(v)
            s_stack = torch.stack(s_list, dim=1)
            v_stack = torch.stack(v_list, dim=1)
            denom = v_stack.sum(dim=1).clamp(min=1.0)
            mean_score = (s_stack * v_stack).sum(dim=1) / denom
            best = mean_score if best is None else torch.maximum(best, mean_score)
        scores.append(best)
    return torch.cat(scores).mean().item()

print("=== Oracle SI-SDR results (test split, true count given -- NOT your real number) ===")
for n_spk in SPEAKER_COUNTS:
    score = evaluate_oracle(models[n_spk], n_spk)
    if score is not None:
        print(f"{n_spk} speakers: {score:.2f} dB SI-SDR")
    else:
        print(f"{n_spk} speakers: no test data found")


In [ ]:
@torch.no_grad()
def separate_file_blind(wav_path, count_model, sep_models, out_prefix="./output/separated_band"):
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
    wav, sr = sf.read(wav_path)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    assert sr == 8000, f"expected 8kHz audio, got {sr}Hz"
    x = torch.from_numpy(wav.astype(np.float32)).unsqueeze(0).to(device)

    n_pred, est = separate_blind(x, count_model, sep_models)[0]
    print(f"Classifier predicted {n_pred} speakers.")

    paths = []
    for c in range(n_pred):
        out_path = f"{out_prefix}_pred{n_pred}spk_s{c}.wav"
        sf.write(out_path, est[c].cpu().numpy(), sr)
        paths.append(out_path)
    print("Saved:", paths)
    return n_pred, paths

demo_meta = next(m for m in metadata if m["n_speakers"] == 3 and "/test/" in m["mixture"])
separate_file_blind(demo_meta["mixture"], count_model, models)
